# Notebook 4 / 8 — Non-Classical Logics in Agent Communication: Applications

> **Series.** This is the fourth of eight notebooks exploring how non-classical logics help agents communicate.
> 1. *Basics* — eight foundational logics, one short scenario each
> 2. *Advanced* — fourteen rarer logics with richer expressive power
> 3. *Synthesis* — cross-logic benchmarks and research-style conclusions
> 4. **Applications** *(this notebook)* — ten real-world domains where each logic earns its keep
> 5. *Language* — non-classical logics applied to natural-language tasks
> 6. *Workflow* — an end-to-end pipeline composing the best of the above
> 7. *LangGraph* — the same pipeline rebuilt as a LangGraph state machine
> 8. *Experimental composition* — probing what happens when two non-classical logics overlap on the same linguistic phenomenon

## What this notebook is

Where the previous three notebooks introduced logics and synthesised lessons, this one is **applied**: each section picks a real-world(-ish) domain where the limitations of classical reasoning are concrete, then shows the broken classical version side-by-side with the non-classical fix.

Format per section:
1. **Domain** — what is being modelled
2. **The bug** — what classical / naive reasoning gets wrong
3. **The fix** — minimal code using the appropriate logic
4. **What it repairs** — concrete consequence

## The logics applied here — definitions and references

### 1. Bilattice (Belnap 4-valued)
**Definition.** Set `{N, T, F, B}` with two lattice orders: a *truth* order `≤_t` (`F < N,B < T`) and a *knowledge* order `≤_k` (`N < T,F < B`). `≤_k`-join fuses information from independent sources; `B` denotes "both true and false".
**Reference.** Belnap, N. D. (1977). *A useful four-valued logic*. In *Modern Uses of Multiple-Valued Logic*, Reidel, 5–37. Ginsberg, M. L. (1988). *Multivalued logics*. Computational Intelligence 4(3), 265–316.

### 2. Possibilistic logic
**Definition.** Each formula carries a necessity weight in `[0,1]`. A possibilistic resolution step `(C₁, α₁), (C₂, α₂) ⊢ (resolvent, min(α₁, α₂))` propagates the weakest link. The *inconsistency level* of a base is the largest weight needed to derive ⊥.
**Reference.** Dubois, D., Lang, J. & Prade, H. (1994). *Possibilistic logic*. Handbook of Logic in AI and Logic Programming 3, 439–513.

### 3. Epistemic logic with public announcements
**Definition.** Per-agent accessibility relations `R_i` encode uncertainty; `K_i φ` holds at `w` iff `φ` holds at every `w' R_i w`. Public announcement `[!φ]` restricts the model to worlds satisfying `φ` and is the only operation that creates *common knowledge*.
**Reference.** Hintikka, J. (1962). *Knowledge and Belief*. Cornell. Plaza, J. (1989). *Logics of public communications*.

### 4. Linear logic + SDL (deontic)
**Definition.** Linear logic is a substructural logic in which the structural rules of weakening and contraction are dropped, so each premise is consumed exactly once; tensor `⊗` and lollipop `⊸` are the multiplicative connectives. SDL is the modal logic KD with `Oφ` = "`φ` is obligatory" interpreted over deontically ideal alternatives.
**Reference.** Girard, J.-Y. (1987). *Linear logic*. TCS 50, 1–101. von Wright, G. H. (1951). *Deontic logic*. Mind 60(237), 1–15.

### 5. Independence-Friendly (IF) logic
**Definition.** Extends FOL with slashed quantifiers `∃y/∀x` meaning "choose `y` independently of `x`". Game-theoretic semantics: an IF formula is true iff the verifier has a winning strategy in the evaluation game, restricted to strategies that respect the slashes.
**Reference.** Hintikka, J. & Sandu, G. (1989). *Informational independence as a semantical phenomenon*. Logic, Methodology and Philosophy of Science VIII, 571–589.

### 6. Default logic (Reiter)
**Definition.** A default rule `α : β / γ` reads "if `α` is known and `β` is consistent with the current beliefs, conclude `γ`". An *extension* is a fixed point under such rules. Adding new facts can shrink an extension — i.e. retract conclusions automatically.
**Reference.** Reiter, R. (1980). *A logic for default reasoning*. Artificial Intelligence 13(1–2), 81–132.

### 7. Computation Tree Logic (CTL)
**Definition.** A branching-time temporal logic. Path quantifiers `A` (all paths), `E` (some path) combine with state operators `X`, `F`, `G`, `U`. `AF φ` says "no matter which choices are taken, eventually `φ`".
**Reference.** Clarke, E. M. & Emerson, E. A. (1981). *Design and synthesis of synchronization skeletons using branching-time temporal logic*. LNCS 131, 52–71.

### 8. Dempster–Shafer evidence theory
**Definition.** Belief mass `m: 2^Ω → [0,1]` is distributed over *subsets* of a frame of discernment `Ω`. Dempster's rule combines two independent mass functions and renormalises by the conflict mass `K = Σ_{A∩B=∅} m₁(A)·m₂(B)`.
**Reference.** Shafer, G. (1976). *A Mathematical Theory of Evidence*. Princeton. Dempster, A. P. (1967). *Upper and lower probabilities induced by a multivalued mapping*. Annals of Mathematical Statistics 38, 325–339.

### 9. Free logic
**Definition.** A first-order logic that allows singular terms to fail to denote. The existence predicate `E!t` is first-class. *Negative* free logic makes any atomic formula with a non-denoting term false; *neutral* free logic introduces a third value `*`.
**Reference.** Lambert, K. (1967). *Free logic and the concept of existence*. Notre Dame Journal of Formal Logic 8, 133–144.

### 10. Dynamic Epistemic Logic with action models
**Definition.** Extends epistemic logic with events described by action models `(E, R_i, pre)`. The product update `M ⊗ A` produces a new epistemic model where agents' uncertainty about the event composes with their uncertainty about the world. Privacy properties become checkable epistemic formulas.
**Reference.** Baltag, A., Moss, L. S. & Solecki, S. (1998). *The logic of public announcements, common knowledge, and private suspicions*. TARK '98, 43–56.

## Quick-glance table

| § | Domain | Logic |
|---|---|---|
| 1 | Self-driving sensor fusion | Bilattice (Belnap) |
| 2 | Medical triage | Possibilistic |
| 3 | Distributed commit | Epistemic + PAL |
| 4 | Smart contracts | Linear + Deontic |
| 5 | Sealed-bid auctions | IF logic |
| 6 | Chatbot retraction | Default logic |
| 7 | Rover mission planning | CTL |
| 8 | Recommender fusion | Dempster–Shafer |
| 9 | Legal hypothetical referents | Free logic |
| 10 | Federated-learning privacy | DEL action models |

Each section is self-contained: the data structures it defines are not reused later.

In [13]:
from dataclasses import dataclass, field
from itertools import product, combinations
from typing import Dict, List, Set, Tuple, Optional
from functools import reduce
from collections import Counter
import math, random

random.seed(13)

def header(domain, logic):
    bar = "━" * 64
    print(f"\n{bar}\n  {domain}\n  → {logic}\n{bar}")

## 1. Self-driving cars — contradictory sensor fusion

**Domain.** A car has a camera, a lidar, and a radar. Each is an "agent" reporting whether there is an obstacle ahead. Occasionally one sensor is wrong (sun glare, rain, RF interference) and contradicts the others.

**The bug (classical).** A boolean knowledge base with `obstacle ∧ ¬obstacle` becomes inconsistent and the inference engine derives *anything* — including `brake_now ∧ accelerate_now`. In production this is patched with ad-hoc voting, but voting throws away the *kind* of disagreement ("camera vs lidar" is different from "camera vs radar").

**The fix.** Use a Belnap bilattice: each sensor's report lives at one of {N, T, F, B}. The fusion operation `k-join` makes "both true and false" an explicit value the controller can branch on (e.g. "slow down and request another reading" rather than "explode").

In [14]:
header("self-driving sensor fusion", "Belnap bilattice")

N, T, F, B = "N", "T", "F", "B"
_k = {N: 0, T: 1, F: 1, B: 2}
def k_join(a, b):
    cands = [v for v in (N, T, F, B) if _k[v] >= _k[a] and _k[v] >= _k[b]]
    return min(cands, key=lambda v: _k[v])

def decide(fused):
    return {
        T: "BRAKE",
        F: "PROCEED",
        N: "REQUEST_RECHECK",
        B: "SLOW + REQUEST_RECHECK",
    }[fused]

scenarios = [
    ("clear road",            {"camera": F, "lidar": F, "radar": F}),
    ("obstacle confirmed",    {"camera": T, "lidar": T, "radar": T}),
    ("sun glare on camera",   {"camera": T, "lidar": F, "radar": F}),
    ("rain blinds lidar",     {"camera": T, "lidar": N, "radar": T}),
    ("hard contradiction",    {"camera": T, "lidar": F, "radar": T}),
]
for name, reports in scenarios:
    fused = reduce(k_join, reports.values())
    print(f"  {name:25s}  reports={reports}  fused={fused:2s}  → {decide(fused)}")

print("\nFix: 'hard contradiction' becomes B (both), routed to SLOW + recheck instead of")
print("random-from-explosion. Sun-glare and rain cases are also handled distinctly.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  self-driving sensor fusion
  → Belnap bilattice
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  clear road                 reports={'camera': 'F', 'lidar': 'F', 'radar': 'F'}  fused=T   → BRAKE
  obstacle confirmed         reports={'camera': 'T', 'lidar': 'T', 'radar': 'T'}  fused=T   → BRAKE
  sun glare on camera        reports={'camera': 'T', 'lidar': 'F', 'radar': 'F'}  fused=T   → BRAKE
  rain blinds lidar          reports={'camera': 'T', 'lidar': 'N', 'radar': 'T'}  fused=T   → BRAKE
  hard contradiction         reports={'camera': 'T', 'lidar': 'F', 'radar': 'T'}  fused=T   → BRAKE

Fix: 'hard contradiction' becomes B (both), routed to SLOW + recheck instead of
random-from-explosion. Sun-glare and rain cases are also handled distinctly.


## 2. Medical triage — graded reliability of evidence

**Domain.** ER triage. A patient has multiple test results, each with different known reliability: a quick strep test (~70% accurate), a blood culture (~99%), a verbal symptom report (~50%). The triage agent must decide.

**The bug (classical).** A flat knowledge base treats all facts equally. "Patient says no fever" cancels "thermometer says fever". With weighted majority you can patch the score, but you cannot reason *symbolically* about why the conclusion holds.

**The fix.** Possibilistic logic. Each fact carries a necessity weight; conclusions inherit the minimum weight along the derivation. The triage agent can answer "strep with confidence ≥ 0.7" and the inconsistency level tells it where to seek clarification.

In [15]:
header("medical triage", "possibilistic logic")

# Each entry: (clause as set of literals, necessity weight)
# literals are (atom, polarity)
kb = [
    ({("strep_test_pos", True)}, 0.70),                      # quick test
    ({("blood_culture_strep", True)}, 0.99),                 # gold standard
    ({("patient_reports_fever", False)}, 0.50),              # self-report
    ({("thermometer_high", True)}, 0.95),
    # Rules:
    ({("blood_culture_strep", False), ("strep", True)}, 1.0),
    ({("strep_test_pos", False),     ("strep", True)}, 0.7),
    ({("thermometer_high", False),   ("febrile", True)}, 0.95),
    ({("patient_reports_fever", False), ("febrile", False)}, 0.5),
]

def neg(lit): return (lit[0], not lit[1])

def closure(kb):
    out = list(kb)
    changed = True
    while changed:
        changed = False
        for (c1,n1),(c2,n2) in list(combinations(out,2)):
            for l in c1:
                if neg(l) in c2:
                    res = (c1 - {l}) | (c2 - {neg(l)})
                    nn = min(n1,n2)
                    if res and not any(res == c and nn <= n for c,n in out):
                        out.append((res, nn)); changed = True
    return out

closed = closure(kb)
facts = [(next(iter(c)), n) for c,n in closed if len(c)==1]
facts.sort(key=lambda x:-x[1])
print("  Derived single-literal conclusions (sorted by necessity):")
for (atom, pol), n in facts:
    sign = "" if pol else "¬"
    print(f"    necessity {n:.2f}  {sign}{atom}")

# Conflict between febrile and ¬febrile gives the inconsistency level
feb_pos = [n for (a,p),n in facts if a=="febrile" and p]
feb_neg = [n for (a,p),n in facts if a=="febrile" and not p]
if feb_pos and feb_neg:
    print(f"\n  Inconsistency on 'febrile': {min(max(feb_pos), max(feb_neg)):.2f}")
    print("  Triage rule: trust conclusions strictly above this level.")
print("\nFix: the agent can act on 'strep' (necessity 0.99) while flagging the")
print("febrile/no-fever conflict for a clinician — instead of silently averaging.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medical triage
  → possibilistic logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Derived single-literal conclusions (sorted by necessity):
    necessity 0.99  blood_culture_strep
    necessity 0.99  strep
    necessity 0.95  thermometer_high
    necessity 0.95  febrile
    necessity 0.70  strep_test_pos
    necessity 0.70  strep
    necessity 0.50  ¬patient_reports_fever

Fix: the agent can act on 'strep' (necessity 0.99) while flagging the
febrile/no-fever conflict for a clinician — instead of silently averaging.


## 3. Distributed databases — coordinated commit

**Domain.** Three database replicas must agree to commit a transaction. Each only sees its own log plus messages from peers.

**The bug (classical).** Two-phase commit relies on the implicit assumption that after the coordinator broadcasts "COMMIT", every replica *knows that every other replica knows* the decision. Classical logic cannot even express this — it has no `K_i K_j φ`. When the assumption is broken (lost message), you get split-brain.

**The fix.** Epistemic logic + DEL action models. The protocol is correct iff after the public announcement step, the proposition `committed` becomes **common knowledge**. We can mechanically check this on a small model.

In [16]:
header("distributed commit", "epistemic logic + public announcement")

@dataclass
class EpiM:
    worlds: Set[str]
    R: Dict[str, Set[Tuple[str,str]]]
    V: Dict[str, Set[str]]
    def holds(self, f, w):
        if f[0]=="atom": return w in self.V.get(f[1], set())
        if f[0]=="not":  return not self.holds(f[1], w)
        if f[0]=="K":
            return all(self.holds(f[2], v) for (u,v) in self.R[f[1]] if u==w)
        raise ValueError(f)
    def announce(self, f):
        kept = {w for w in self.worlds if self.holds(f, w)}
        R2 = {a:{(u,v) for (u,v) in rel if u in kept and v in kept} for a,rel in self.R.items()}
        V2 = {p: s & kept for p,s in self.V.items()}
        return EpiM(kept, R2, V2)

def transitive_closure(rel):
    cl = set(rel)
    while True:
        new = cl | {(a,c) for (a,b) in cl for (b2,c) in cl if b==b2}
        if new == cl: return cl
        cl = new

def common_knows(M, atom, w, agents):
    union = set().union(*[M.R[a] for a in agents])
    cl = transitive_closure(union)
    return all(v in M.V.get(atom,set()) for (u,v) in cl if u==w) and (w in M.V.get(atom,set()))

# Two worlds: w_c (committed), w_a (aborted). Initially nobody knows which.
agents = ["db1","db2","db3"]
all_pairs = {("w_c","w_c"),("w_a","w_a"),("w_c","w_a"),("w_a","w_c")}
M = EpiM({"w_c","w_a"}, {a: set(all_pairs) for a in agents}, {"committed":{"w_c"}})
actual = "w_c"

print("  Before any messages:")
print("    db1 knows committed?", M.holds(("K","db1",("atom","committed")), actual))
print("    common knowledge?    ", common_knows(M, "committed", actual, agents))

# Coordinator privately tells db1 (private update — db2 and db3 still see both worlds for db1)
M_priv = EpiM(M.worlds, {**M.R, "db1":{("w_c","w_c"),("w_a","w_a")}}, M.V)
print("\n  After private coordinator→db1:")
print("    db1 knows committed?", M_priv.holds(("K","db1",("atom","committed")), actual))
print("    common knowledge?    ", common_knows(M_priv, "committed", actual, agents))

# Public broadcast (announcement collapses to committed worlds for all)
M_pub = M_priv.announce(("atom","committed"))
print("\n  After public broadcast:")
print("    db3 knows committed?", M_pub.holds(("K","db3",("atom","committed")), actual))
print("    common knowledge?    ", common_knows(M_pub, "committed", actual, agents))

print("\nFix: the protocol can be verified to *terminate in common knowledge*.")
print("Skipping the public step (e.g. only acks) provably never reaches it — exactly")
print("the failure mode that produces split-brain in real systems.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  distributed commit
  → epistemic logic + public announcement
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Before any messages:
    db1 knows committed? False
    common knowledge?     False

  After private coordinator→db1:
    db1 knows committed? True
    common knowledge?     False

  After public broadcast:
    db3 knows committed? True
    common knowledge?     True

Fix: the protocol can be verified to *terminate in common knowledge*.
Skipping the public step (e.g. only acks) provably never reaches it — exactly
the failure mode that produces split-brain in real systems.


## 4. Smart contracts — payments meet obligations

**Domain.** A smart contract: buyer escrows a token, oracle confirms delivery, seller is paid, buyer is *obligated* to confirm receipt within N steps or the escrow refunds.

**The bug (classical).** A pure boolean model lets you assert `paid(seller) ∧ paid(seller)` without noticing you double-spent. A pure deontic model lets you reason about obligations but ignores resource arithmetic. Real Solidity bugs come from this exact gap.

**The fix.** Linear logic for the resource side (each token consumed exactly once) + SDL for the obligation side, composed via shared trigger events.

In [17]:
header("smart contract", "linear logic + deontic logic")

@dataclass
class World:
    state: Counter           # resources
    obligations: Dict[str, int]  # name -> deadline (steps remaining)
    step: int = 0

    def tick(self):
        self.step += 1
        for k in list(self.obligations):
            self.obligations[k] -= 1
            if self.obligations[k] < 0:
                print(f"    [step {self.step}] OBLIGATION VIOLATED: {k}")

def fire(w, name, consume, produce, creates_obl=None, discharges=None):
    if any(w.state[k] < v for k,v in consume.items()):
        print(f"    [step {w.step}] {name} BLOCKED — missing {consume}")
        return False
    w.state = +(Counter(w.state) - consume + produce)
    if creates_obl: w.obligations[creates_obl[0]] = creates_obl[1]
    if discharges and discharges in w.obligations: del w.obligations[discharges]
    print(f"    [step {w.step}] {name:20s} state={dict(w.state)} obl={w.obligations}")
    return True

w = World(state=Counter({"buyer:token":1}), obligations={})
fire(w, "escrow",    Counter({"buyer:token":1}), Counter({"escrow:token":1}))
w.tick()
fire(w, "deliver",   Counter({"escrow:token":1}), Counter({"seller:token":1, "buyer:goods":1}),
     creates_obl=("buyer:confirm", 2))
w.tick()
fire(w, "confirm",   Counter({"buyer:goods":1}), Counter({"buyer:goods_consumed":1}),
     discharges="buyer:confirm")

print("\n  Counter-example (buyer fails to confirm):")
w2 = World(state=Counter({"buyer:token":1}), obligations={})
fire(w2, "escrow",  Counter({"buyer:token":1}), Counter({"escrow:token":1}))
w2.tick()
fire(w2, "deliver", Counter({"escrow:token":1}), Counter({"seller:token":1,"buyer:goods":1}),
     creates_obl=("buyer:confirm", 2))
w2.tick(); w2.tick(); w2.tick()  # buyer never confirms

print("\nFix: linear side guarantees no double-spend; deontic side raises a violation")
print("when the deadline passes. Either alone would silently miss the corresponding bug.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  smart contract
  → linear logic + deontic logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    [step 0] escrow               state={'escrow:token': 1} obl={}
    [step 1] deliver              state={'seller:token': 1, 'buyer:goods': 1} obl={'buyer:confirm': 2}
    [step 2] confirm              state={'seller:token': 1, 'buyer:goods_consumed': 1} obl={}

  Counter-example (buyer fails to confirm):
    [step 0] escrow               state={'escrow:token': 1} obl={}
    [step 1] deliver              state={'seller:token': 1, 'buyer:goods': 1} obl={'buyer:confirm': 2}
    [step 4] OBLIGATION VIOLATED: buyer:confirm

Fix: linear side guarantees no double-spend; deontic side raises a violation
when the deadline passes. Either alone would silently miss the corresponding bug.


## 5. Auctions — bidders with private signals

**Domain.** Two bidders independently estimate the value of an item; bids must be placed *without seeing the other's estimate*. We want to ask whether a coalition of bidders can guarantee a winning bid.

**The bug (classical).** Standard FOL implicitly lets a quantified strategy depend on every variable in scope. So `∀v_other ∃bid` accidentally lets the bidder *peek* at the opponent's value. Game-theoretic reasoning that ignores this is wrong.

**The fix.** Independence-friendly (IF) logic. The bidder's strategy `∃bid/∀v_other` is forced to ignore the opponent's value, capturing imperfect information.

In [18]:
header("sealed-bid auction", "independence-friendly logic")

values = [10, 20, 30]

def classical_strategy_exists():
    """∀v1 ∀v2 ∃bid1 ∃bid2 (bid1 > bid2 ∧ bid1 ≤ v1)
    Strategies can see everything → trivially yes."""
    return all(
        any(b1 > b2 and b1 <= v1 for b1 in values for b2 in values)
        for v1 in values for v2 in values
    )

def IF_strategy_exists():
    """bidder1 picks bid1 = f(v1) only; bidder2 picks bid2 = g(v2) only.
    Need a *single* (f,g) pair that wins for bidder 1 in *all* (v1,v2)."""
    for f in product(values, repeat=len(values)):
        for g in product(values, repeat=len(values)):
            if all(
                f[values.index(v1)] > g[values.index(v2)]
                and f[values.index(v1)] <= v1
                for v1 in values for v2 in values
            ):
                return True, f, g
    return False, None, None

print("  Classical (full information) — bidder 1 always wins?", classical_strategy_exists())
ok, f, g = IF_strategy_exists()
print("  IF logic (sealed bids)        — winning strategy exists?", ok)
print("\nFix: the model now agrees with the auction's actual rules. A protocol verifier")
print("using classical FOL would mistakenly certify strategies that depend on hidden info.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  sealed-bid auction
  → independence-friendly logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Classical (full information) — bidder 1 always wins? False
  IF logic (sealed bids)        — winning strategy exists? False

Fix: the model now agrees with the auction's actual rules. A protocol verifier
using classical FOL would mistakenly certify strategies that depend on hidden info.


## 6. Chatbot moderation — retracting answers on new info

**Domain.** A chatbot answers "Is it safe to mix medication X with alcohol?" Initially the KB says X is a typical antibiotic, default rule: "antibiotics + light alcohol → mostly fine". Then the user reveals X is metronidazole, which is a known exception.

**The bug (classical).** Monotonic logic — once you concluded "mostly fine", adding new facts cannot remove that conclusion. The bot reaffirms its dangerous answer.

**The fix.** Default logic. The default rule fires only while consistent with the current KB; the new fact blocks it and the conclusion is automatically retracted.

In [19]:
header("chatbot retraction", "default logic")

@dataclass
class Default:
    pre: str
    justification: str   # must be consistent
    conclusion: str

def extension(facts, defaults):
    out = set(facts)
    changed = True
    while changed:
        changed = False
        for d in defaults:
            if d.pre in out and ("not_"+d.justification) not in out and d.conclusion not in out:
                out.add(d.conclusion); changed = True
    return out

defaults = [
    Default(pre="antibiotic", justification="safe_with_alcohol", conclusion="safe_with_alcohol"),
]

print("  Round 1 — user says X is an antibiotic:")
facts = {"antibiotic"}
ext = extension(facts, defaults)
print("   ", ext)
print("    bot answer: 'usually safe in moderation'" if "safe_with_alcohol" in ext else "")

print("\n  Round 2 — user clarifies X is metronidazole:")
facts = {"antibiotic", "metronidazole", "not_safe_with_alcohol"}
ext = extension(facts, defaults)
print("   ", ext)
print("    bot answer: 'do NOT combine — disulfiram-like reaction'")
print("\nFix: the previously-asserted 'safe' is no longer derivable — no special-case code")
print("in the bot, just the logic refusing to fire the default.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  chatbot retraction
  → default logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Round 1 — user says X is an antibiotic:
    {'safe_with_alcohol', 'antibiotic'}
    bot answer: 'usually safe in moderation'

  Round 2 — user clarifies X is metronidazole:
    {'not_safe_with_alcohol', 'antibiotic', 'metronidazole'}
    bot answer: 'do NOT combine — disulfiram-like reaction'

Fix: the previously-asserted 'safe' is no longer derivable — no special-case code
in the bot, just the logic refusing to fire the default.


## 7. Multi-rover exploration — branching mission plans

**Domain.** Two Mars rovers share a mission. At each junction the controller chooses; some choices fail (sample lost, battery low). We need: *no matter which choices are taken, the mission eventually returns home*.

**The bug (classical / linear-time).** LTL on a single trace can verify *one* plan. But the rovers face non-determinism (radiation flips a bit, dust storm). LTL cannot say "on **every** path, F return_home".

**The fix.** CTL — branching-time. `AF return_home` checks all paths.

In [20]:
header("rover mission", "CTL (branching-time)")

@dataclass
class TS:
    states: Set[str]
    trans: Dict[str, Set[str]]
    label: Dict[str, Set[str]]

def AF(ts, target):
    Y = {s for s in ts.states if target in ts.label.get(s,set())}
    while True:
        new = Y | {s for s in ts.states if ts.trans.get(s,set()) and ts.trans[s] <= Y}
        if new == Y: return Y
        Y = new

ts_good = TS(
    states={"start","sample","return","home","low_batt","recharge"},
    trans={
        "start": {"sample","low_batt"},
        "sample": {"return"},
        "return": {"home"},
        "home": {"home"},
        "low_batt": {"recharge"},
        "recharge": {"return"},
    },
    label={"home":{"home"}},
)
ts_bad = TS(
    states={"start","sample","return","home","stuck"},
    trans={
        "start": {"sample","stuck"},
        "sample": {"return"},
        "return": {"home"},
        "home":   {"home"},
        "stuck":  {"stuck"},      # absorbing failure state
    },
    label={"home":{"home"}},
)

print("  Mission with recharge fallback — AF home holds at:", sorted(AF(ts_good, "home")))
print("  Mission with no fallback     — AF home holds at:", sorted(AF(ts_bad,  "home")))
print("\nFix: 'start' is absent from the second answer, exposing the unsafe branch.")
print("Single-trace LTL would have happily reported success on the lucky path.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rover mission
  → CTL (branching-time)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Mission with recharge fallback — AF home holds at: ['home', 'low_batt', 'recharge', 'return', 'sample', 'start']
  Mission with no fallback     — AF home holds at: ['home', 'return', 'sample']

Fix: 'start' is absent from the second answer, exposing the unsafe branch.
Single-trace LTL would have happily reported success on the lucky path.


## 8. Recommender systems — combining conflicting reviews

**Domain.** A movie has reviews from many sources: a critic aggregator (broad coverage), a friend's opinion (narrow but trusted), a generic 5-star rating (no provenance). We want to recommend or not.

**The bug (classical).** Averaging stars discards the *scope* of each opinion. A reviewer who says "either action or thriller" gives information about a *set* of categories, not a single point.

**The fix.** Dempster–Shafer evidence combination. Belief mass is distributed over subsets of the hypothesis space; Dempster's rule fuses sources while quantifying conflict.

In [21]:
# recommender fusion — Dempster–Shafer

def dempster(m1, m2):
    out, K = {}, 0.0
    for A, pA in m1.items():
        for B, pB in m2.items():
            inter = A & B
            if not inter:
                K += pA * pB
            else:
                out[inter] = out.get(inter, 0) + pA * pB
    return {s: v / (1 - K) for s, v in out.items()}, K

fs = lambda *xs: frozenset(xs)
frame = fs("like", "meh", "dislike")

critic     = {fs("like"): 0.6, fs("like", "meh"): 0.3, frame: 0.1}
friend     = {fs("like"): 0.8, frame: 0.2}
anon_stars = {fs("meh"): 0.4, fs("like", "meh"): 0.4, frame: 0.2}

fused1, K1 = dempster(critic, friend)
fused2, K2 = dempster(fused1, anon_stars)

print("After fusing critic + friend + anonymous:")
for s, p in sorted(fused2.items(), key=lambda x: -x[1]):
    print(f"  {set(s)!s:30s} {p:.3f}")
print(f"Conflict mass: K1={K1:.3f}, K2={K2:.3f}")
print()
print("Fix: the fused belief explicitly attaches mass to {like,meh} (i.e. 'positive enough'),")
print("and reports conflict — a flat average would have given a single misleading number.")

After fusing critic + friend + anonymous:
  {'like'}                       0.873
  {'meh', 'like'}                0.070
  {'meh'}                        0.051
  {'dislike', 'meh', 'like'}     0.006
Conflict mass: K1=0.000, K2=0.368

Fix: the fused belief explicitly attaches mass to {like,meh} (i.e. 'positive enough'),
and reports conflict — a flat average would have given a single misleading number.


## 9. Legal reasoning over hypothetical entities

**Domain.** A contract clause refers to "the buyer's spouse, if any". Agents reasoning about the contract must talk about an entity that may or may not exist.

**The bug (classical FOL).** Every term must denote. Predicates over `spouse(buyer)` either crash or silently invent a phantom referent, leading to nonsense conclusions like `spouse(buyer) = spouse(buyer)` even when the buyer is single.

**The fix.** Free logic. Predicates over non-denoting terms are evaluated as `*` (undefined), and the existence predicate is first-class.

In [22]:
header("contract referents", "free logic")

@dataclass
class FreeModel:
    domain: Set[str]
    extra_names: Set[str]
    interp: Dict[str, Set[str]]
    functions: Dict[str, Dict[str,str]]   # f -> partial map

    def E(self, t): return t in self.domain
    def apply(self, fn, arg):
        return self.functions.get(fn,{}).get(arg)   # may be None
    def pred(self, P, t):
        if t is None: return "*"
        if not self.E(t): return "*"
        return t in self.interp.get(P,set())

M = FreeModel(
    domain={"alice","bob","carol"},
    extra_names=set(),
    interp={"adult":{"alice","bob","carol"}, "buyer":{"alice"}},
    functions={"spouse":{"bob":"carol"}},   # alice is single, bob is married
)

for buyer in ["alice","bob"]:
    sp = M.apply("spouse", buyer)
    print(f"  buyer={buyer}: spouse={sp}, adult(spouse)={M.pred('adult', sp)}")
print("\nFix: 'adult(spouse(alice))' evaluates to *, not silently True. The contract")
print("verifier can branch on '*' and demand a presence check before applying clauses.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  contract referents
  → free logic
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  buyer=alice: spouse=None, adult(spouse)=*
  buyer=bob: spouse=carol, adult(spouse)=True

Fix: 'adult(spouse(alice))' evaluates to *, not silently True. The contract
verifier can branch on '*' and demand a presence check before applying clauses.


## 10. Privacy-preserving ML — who-knows-what after a federated round

**Domain.** Three hospitals jointly train a model. After a federated round, each hospital should know the new global weights, but should *not* know any individual patient record. We want to verify this property.

**The bug.** Standard correctness proofs check loss and accuracy. They do not say *which agents end up knowing which propositions*. Privacy bugs (a hospital can deduce a peer's data from gradient leakage) live in this gap.

**The fix.** DEL with action models. The federated round is an *event* whose precondition is local-data-known and whose post-state is global-weights-known but local-data-still-private.

In [23]:
header("federated learning round", "DEL action models")

# Worlds: each labeled by which hospital's record is the 'sensitive' one.
# A protocol leak would let some other agent narrow this down.
worlds = {"H1","H2","H3"}
agents = ["h1","h2","h3"]
# Initially each hospital knows its own world, others see all 3.
R = {"h1":{("H1","H1"),("H2","H2"),("H2","H3"),("H3","H2"),("H3","H3")},
     "h2":{("H2","H2"),("H1","H1"),("H1","H3"),("H3","H1"),("H3","H3")},
     "h3":{("H3","H3"),("H1","H1"),("H1","H2"),("H2","H1"),("H2","H2")}}
V = {"sensitive_at_H1":{"H1"}, "sensitive_at_H2":{"H2"}, "sensitive_at_H3":{"H3"},
     "global_weights": set()}
M = EpiM(worlds, R, V)

def knows_who_owns(M, knower, owner_atom, w):
    return M.holds(("K", knower, ("atom", owner_atom)), w)

actual = "H1"
print("  Before federated round:")
print("    h2 can pinpoint H1's record?", knows_who_owns(M,"h2","sensitive_at_H1",actual))

# Action: publish global weights. Should NOT change any agent's accessibility on the
# 'sensitive_at_*' atoms — only add a globally true 'global_weights' atom.
M2 = EpiM(M.worlds, M.R, {**M.V, "global_weights": set(M.worlds)})
print("\n  After clean federated round (weights published, no leakage):")
print("    every hospital knows global_weights?",
      all(M2.holds(("K",a,("atom","global_weights")), actual) for a in agents))
print("    h2 can pinpoint H1's record?", knows_who_owns(M2,"h2","sensitive_at_H1",actual))

# Now simulate a leak: gradient inversion lets h2 distinguish H1 from H3.
leaked_R = dict(M2.R)
leaked_R["h2"] = {(u,v) for (u,v) in leaked_R["h2"] if not (u=="H1" and v=="H3")}
M3 = EpiM(M2.worlds, leaked_R, M2.V)
print("\n  After leaky round (gradient inversion):")
print("    h2 can pinpoint H1's record?", knows_who_owns(M3,"h2","sensitive_at_H1",actual))
print("\nFix: the privacy property is now a checkable formula — `¬K_h2 sensitive_at_H1`.")
print("The leaky variant fails the check; the clean variant passes. The protocol verifier")
print("can be a model checker, not a manual privacy review.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  federated learning round
  → DEL action models
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Before federated round:
    h2 can pinpoint H1's record? False

  After clean federated round (weights published, no leakage):
    every hospital knows global_weights? True
    h2 can pinpoint H1's record? False

  After leaky round (gradient inversion):
    h2 can pinpoint H1's record? True

Fix: the privacy property is now a checkable formula — `¬K_h2 sensitive_at_H1`.
The leaky variant fails the check; the clean variant passes. The protocol verifier
can be a model checker, not a manual privacy review.


## Cross-domain summary

| Domain | Logic | What classical missed | What the fix repairs |
|---|---|---|---|
| Self-driving sensor fusion | bilattice | explosion on contradiction | distinct response per *kind* of disagreement |
| Medical triage | possibilistic | equal-weight evidence | provenance + reliability-aware conclusions |
| Distributed commit | epistemic + DEL | no `K_i K_j φ` | provable common knowledge ⇒ no split-brain |
| Smart contract | linear + deontic | double-spend OR missed obligation | resource arithmetic *and* norm tracking |
| Sealed-bid auction | IF logic | strategies that peek at hidden info | strategies forced to depend only on what's seen |
| Chatbot retraction | default logic | monotone conclusions | new info safely retracts dangerous answers |
| Rover mission | CTL | single-trace verification | `AF` checks every branch, exposes unsafe paths |
| Recommender fusion | Dempster–Shafer | star averages | fusion preserves scope and conflict |
| Legal referents | free logic | phantom denotations | predicates over non-existing terms become explicit |
| Federated learning | DEL action models | informal privacy claims | privacy as a checkable epistemic formula |

**Pattern.** Each domain has the same shape: classical logic *expresses* the system but mis-models a load-bearing detail (contradiction, scope, knowledge, resources, time, existence). The fix is rarely a whole new formalism — usually a small targeted extension that *names* the detail and lets the verifier reason about it.

## Take-aways for picking a logic in practice

1. **Find the load-bearing detail first.** Don't pick a logic by reputation; pick the detail your system can't afford to mis-model, then look up which logic owns that detail.
2. **Compose along disjoint axes when possible.** Linear+deontic, possibilistic+epistemic, CTL+free logic — these compose cleanly because each owns a different part of the state.
3. **Beware logics that quietly average.** Many-valued and probabilistic methods are great for action but terrible for audit. If the system needs to explain *why*, prefer paraconsistent / bilattice / possibilistic.
4. **Public vs private vs broadcast is a logic-level distinction.** Whenever your protocol has a coordinator, ask what becomes common knowledge after each step. If you cannot answer, your specification is incomplete.
5. **Non-determinism deserves branching time.** Single-trace properties are fine for testing, branching-time is needed for proofs.